# Driver Ranking Analysis

This notebook analyzes F1 driver performance across different weather conditions, comparing their overall rankings and how race outcomes change for rainy and dry races. We create multiple data aggregations and visualizations to identify which drivers excel in adverse weather conditions.

In [2]:
%pip install pandas
%pip install numpy
%pip install altair

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
#Importing necessary libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Load and Prepare Data

Let us load the F1 lap weather dataset created earlier and filter it to include only clean laps (no pit laps, terminal laps, or safety car periods). This is done to ensure that we're analyzing legitimate racing performance data without any outliers.

In [4]:
# Load the lap-weather dataset from Luis
df = pd.read_pickle('../data/f1_lap_weather_data.pkl')

print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)

Data shape: (188482, 41)

Columns: ['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'TrackStatus', 'Position', 'FastF1Generated', 'IsAccurate', 'Year', 'Location', 'EventName', 'LapStartTimeUTC', 'IsPitLap', 'IsTerminalLap', 'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindDirection', 'WindSpeed', 'Rainfall']

Data types:
Time                  timedelta64[ns]
Driver                         object
DriverNumber                    int64
LapTime               timedelta64[ns]
LapNumber                     float64
Stint                         float64
PitOutTime            timedelta64[ns]
PitInTime             timedelta64[ns]
Sector1Time           timedelta64[ns]
Sector2Time           timedelta64[ns]
Sector3Time 

In [5]:
# Check data quality
print(f"Unique races: {df['EventName'].nunique()}")
print(f"Unique drivers: {df['Driver'].nunique()}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
print(f"\nPosition column null count: {df['Position'].isna().sum()} / {len(df)}")
print(f"Position data type: {df['Position'].dtype}")
df[['Driver', 'EventName', 'Year', 'LapNumber', 'Position', 'Team']].head(10)

Unique races: 36
Unique drivers: 43
Year range: 2018 - 2025

Position column null count: 2 / 188482
Position data type: float64


,Driver,EventName,Year,LapNumber,Position,Team
0,GAS,Australian Grand Prix,2018,1.0,17.0,Racing Bulls
1,GAS,Australian Grand Prix,2018,2.0,17.0,Racing Bulls
2,GAS,Australian Grand Prix,2018,3.0,17.0,Racing Bulls
3,GAS,Australian Grand Prix,2018,4.0,17.0,Racing Bulls
4,GAS,Australian Grand Prix,2018,5.0,17.0,Racing Bulls
5,GAS,Australian Grand Prix,2018,6.0,16.0,Racing Bulls
6,GAS,Australian Grand Prix,2018,7.0,16.0,Racing Bulls
7,GAS,Australian Grand Prix,2018,8.0,16.0,Racing Bulls
8,GAS,Australian Grand Prix,2018,9.0,16.0,Racing Bulls
9,GAS,Australian Grand Prix,2018,10.0,16.0,Racing Bulls


In [6]:

print(f"Total laps (original): {len(df)}")

Total laps (original): 188482


In [7]:
# Filter to clean racing laps only
df = df[
    (df['TrackStatus'] == 1)
    & (df['IsPitLap'] == False)
    & (df['IsTerminalLap'] == False)
]
# cast Position as numeric
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')

# unique race identifier
df['RaceID'] = df['EventName'] + ' ' + df['Year'].astype(str)


print(f"Clean laps: {len(df)}")
print(f"Total Races: {df['RaceID'].nunique()}")


Clean laps: 156243
Total Races: 172


## Create Race-Level Summary

Let us aggregate lap data to the race level and add key features to the data (finish position, position changes and overtaking opportunities). This gives us a bird's-eye view of driver performance across entire races.

In [8]:
race_results_list = []

for race_id in df['RaceID'].unique():
    race_data = df[df['RaceID'] == race_id]
    
    # Get unique drivers in a race
    drivers = race_data[['Driver', 'DriverNumber', 'Team']].drop_duplicates()
    
    for _, driver_row in drivers.iterrows():
        driver = driver_row['Driver']
        driver_race = race_data[race_data['Driver'] == driver].copy()
        
        if len(driver_race) == 0:
            continue
            
        # Extract race info
        event_name = race_data['EventName'].iloc[0]
        year = race_data['Year'].iloc[0]
        location = race_data['Location'].iloc[0]
        
        # Position tracking
        valid_positions = driver_race['Position'].dropna()
        
        if len(valid_positions) == 0:
            continue
        
        # Grid position (starting lap position)
        first_lap = driver_race[driver_race['LapNumber'] == 1.0]
        grid_pos = first_lap['Position'].iloc[0] if len(first_lap) > 0 and pd.notna(first_lap['Position'].iloc[0]) else np.nan
        
        # Finishing position (last valid position)
        finishing_pos = valid_positions.iloc[-1]
        
        # Position statistics
        best_pos = valid_positions.min()
        worst_pos = valid_positions.max()
        avg_pos = valid_positions.mean()
        pos_std = valid_positions.std()
        
        # Positions gained/lost 
        if pd.notna(grid_pos):
            positions_gained = grid_pos - finishing_pos
        else:
            positions_gained = np.nan
        
        # Laps led
        laps_led = len(driver_race[driver_race['Position'] == 1.0])
        
        # Pit stops
        pit_stops = len(driver_race[driver_race['IsPitLap'] == True])
        
        # DNF detection 
        max_laps_any_driver = race_data['LapNumber'].max()
        driver_max_laps = driver_race['LapNumber'].max()
        dnf = driver_max_laps < (max_laps_any_driver - 2)  # Allow 2 lap tolerance
        
        race_results_list.append({
            'Driver': driver,
            'DriverNumber': driver_row['DriverNumber'],
            'Team': driver_row['Team'],
            'EventName': event_name,
            'Year': year,
            'Location': location,
            'GridPosition': grid_pos,
            'FinishingPosition': finishing_pos,
            'BestPosition': best_pos,
            'WorstPosition': worst_pos,
            'AvgPosition': avg_pos,
            'PositionStdDev': pos_std,
            'PositionsGained': positions_gained,
            'LapsLed': laps_led,
            'PitStops': pit_stops,
            'LapsCompleted': int(driver_max_laps),
            'DNF': dnf
        })

race_results_df = pd.DataFrame(race_results_list)
print(f"\nData with new features for the first race:")
race_results_df.head(20)


Data with new features for the first race:


,Driver,DriverNumber,Team,EventName,Year,Location,GridPosition,FinishingPosition,BestPosition,WorstPosition,AvgPosition,PositionStdDev,PositionsGained,LapsLed,PitStops,LapsCompleted,DNF
0,GAS,10,Racing Bulls,Australian Grand Prix,2018,Melbourne,17.0,16.0,16.0,17.0,16.444444,0.527046,1.0,0,0,13,True
1,PER,11,Aston Martin,Australian Grand Prix,2018,Melbourne,12.0,11.0,10.0,12.0,11.340909,0.525763,1.0,0,0,57,False
2,ALO,14,McLaren,Australian Grand Prix,2018,Melbourne,10.0,5.0,5.0,10.0,6.954545,2.420403,5.0,0,0,57,False
3,LEC,16,Kick Sauber,Australian Grand Prix,2018,Melbourne,18.0,13.0,13.0,18.0,14.452381,1.940611,5.0,0,0,57,False
4,STR,18,Williams,Australian Grand Prix,2018,Melbourne,14.0,14.0,14.0,15.0,14.318182,0.471155,0.0,0,0,57,False
5,VAN,2,McLaren,Australian Grand Prix,2018,Melbourne,11.0,9.0,8.0,11.0,9.627907,1.069563,2.0,0,0,57,False
6,MAG,20,Haas,Australian Grand Prix,2018,Melbourne,4.0,4.0,4.0,4.0,4.000000,0.000000,0.0,0,0,21,True
7,HUL,27,Alpine,Australian Grand Prix,2018,Melbourne,7.0,7.0,6.0,8.0,7.022727,0.263133,0.0,0,0,57,False
8,HAR,28,Racing Bulls,Australian Grand Prix,2018,Melbourne,NaN,15.0,15.0,20.0,15.975000,1.458617,NaN,0,0,56,False
9,RIC,3,Red Bull Racing,Australian Grand Prix,2018,Melbourne,8.0,4.0,4.0,8.0,5.022727,1.372295,4.0,0,0,57,False


In [9]:
print("race_results_df summary:")
print(race_results_df.describe())
print(f"\nDNF count: {race_results_df['DNF'].sum()}")

race_results_df summary:
       DriverNumber         Year  GridPosition  FinishingPosition  \
count   3320.000000  3320.000000   1608.000000        3320.000000   
mean      27.977108  2021.665964     10.017413           9.538554   
std       24.595849     2.302556      5.542140           5.211440   
min        1.000000  2018.000000      1.000000           1.000000   
25%       10.000000  2020.000000      5.000000           5.000000   
50%       20.000000  2022.000000     10.000000          10.000000   
75%       44.000000  2024.000000     15.000000          14.000000   
max       99.000000  2025.000000     20.000000          20.000000   

       BestPosition  WorstPosition  AvgPosition  PositionStdDev  \
count   3320.000000    3320.000000  3320.000000     3290.000000   
mean       7.239157      12.715663     9.823600        1.569533   
std        4.545379       5.599886     5.082858        0.987460   
min        1.000000       1.000000     1.000000        0.000000   
25%        3.00000

## Create Driver-Level Statistics

Let us now aggregate data for each driver. We will compute their statistics across all races in our dataset. This will use the new features we created earlier(like average finish positions, total position changes etc.) to compare overall performance of drivers.

In [10]:

driver_stats_list = []

for driver in race_results_df['Driver'].unique():
    driver_races = race_results_df[race_results_df['Driver'] == driver]
    
    # Driver info
    driver_num = driver_races['DriverNumber'].iloc[0]
    team = driver_races['Team'].iloc[0]
    
    # Race counts
    races_entered = len(driver_races)
    races_completed = len(driver_races[driver_races['DNF'] == False])
    dnf_count = races_entered - races_completed
    
    # Position statistics
    avg_grid_pos = driver_races['GridPosition'].mean()
    avg_finish_pos = driver_races['FinishingPosition'].mean()
    
    # Position gained/lost
    avg_positions_gained = driver_races['PositionsGained'].mean()
    races_with_gains = len(driver_races[driver_races['PositionsGained'] > 0])
    races_with_losses = len(driver_races[driver_races['PositionsGained'] < 0])
    
    # Driver Performance metrics
    podiums = len(driver_races[driver_races['FinishingPosition'] <= 3])
    poles = len(driver_races[driver_races['GridPosition'] == 1.0])
    wins = len(driver_races[driver_races['FinishingPosition'] == 1.0])
    total_laps_led = driver_races['LapsLed'].sum()
    
    # DriverConsistency
    avg_position_std = driver_races['PositionStdDev'].mean()
    
    driver_stats_list.append({
        'Driver': driver,
        'DriverNumber': driver_num,
        'Team': team,
        'RacesEntered': races_entered,
        'RacesCompleted': races_completed,
        'DNFCount': dnf_count,
        'DNFRate': dnf_count / races_entered if races_entered > 0 else 0,
        'AvgGridPosition': avg_grid_pos,
        'AvgFinishPosition': avg_finish_pos,
        'AvgPositionsGained': avg_positions_gained,
        'RacesWithGains': races_with_gains,
        'RacesWithLosses': races_with_losses,
        'Podiums': podiums,
        'Poles': poles,
        'Wins': wins,
        'TotalLapsLed': total_laps_led,
        'AvgPositionConsistency': avg_position_std
    })

driver_stats_df = pd.DataFrame(driver_stats_list).sort_values('RacesEntered', ascending=False)
print(f"Total drivers: {driver_stats_df.shape[0]}")
print(f"\nTop 10 drivers by order of races entered:")
driver_stats_df.head(10)

Total drivers: 43

Top 10 drivers by order of races entered:


,Driver,DriverNumber,Team,RacesEntered,RacesCompleted,DNFCount,DNFRate,AvgGridPosition,AvgFinishPosition,AvgPositionsGained,RacesWithGains,RacesWithLosses,Podiums,Poles,Wins,TotalLapsLed,AvgPositionConsistency
13,HAM,44,Mercedes,168,160,8,0.047619,5.273810,4.136905,1.107143,39,18,87,16,40,1632,1.182980
11,VER,33,Red Bull Racing,166,155,11,0.066265,3.258824,2.728916,0.670588,31,16,123,28,69,3140,0.889934
0,GAS,10,Racing Bulls,164,146,18,0.109756,10.714286,10.646341,0.246753,38,31,2,0,1,25,1.809341
3,LEC,16,Kick Sauber,163,146,17,0.104294,5.650000,5.723926,-0.125000,27,34,55,10,12,755,1.370370
4,STR,18,Williams,163,142,21,0.128834,12.734177,11.711656,0.911392,34,27,2,0,0,22,1.893972
15,SAI,55,Alpine,161,148,13,0.080745,7.475610,7.173913,0.390244,34,27,26,2,4,255,1.484522
23,RUS,63,Williams,151,134,17,0.112583,9.095890,8.900662,0.712329,40,18,24,5,6,342,1.380727
22,NOR,4,McLaren,150,139,11,0.073333,6.466667,6.473333,0.186667,31,24,48,4,12,606,1.360123
17,BOT,77,Mercedes,146,128,18,0.123288,9.083333,8.321918,0.305556,23,28,46,7,8,405,1.332481
1,PER,11,Aston Martin,142,129,13,0.091549,8.044776,6.957746,1.537313,39,14,39,4,6,295,1.564991


In [11]:
# Drivers with most position gains
print("\nTop 10 drivers by average positions gained per race:")
print(driver_stats_df.nlargest(10, 'AvgPositionsGained')[['Driver', 'Team', 'AvgPositionsGained', 'RacesEntered']])

print("\nTop 10 drivers by maximum wins:")
print(driver_stats_df.nlargest(10, 'Wins')[['Driver', 'Team', 'Wins', 'Podiums', 'Poles']])


Top 10 drivers by average positions gained per race:
   Driver             Team  AvgPositionsGained  RacesEntered
5     VAN          McLaren            5.285714            20
19    ERI      Kick Sauber            2.750000            20
21    KVY     Racing Bulls            2.235294            38
9     RIC  Red Bull Racing            2.123077           124
26    LAT         Williams            1.678571            59
8     HAR     Racing Bulls            1.666667            18
1     PER     Aston Martin            1.537313           142
20    ALB     Racing Bulls            1.193548           123
40    ANT         Mercedes            1.166667            23
13    HAM         Mercedes            1.107143           168

Top 10 drivers by maximum wins:
   Driver             Team  Wins  Podiums  Poles
11    VER  Red Bull Racing    69      123     28
13    HAM         Mercedes    40       87     16
3     LEC      Kick Sauber    12       55     10
22    NOR          McLaren    12       48     

## Create Lap-by-Lap Position Tracking

Let us track each driver's position changes across laps in a race. This granular data will later allow us to compare driver performance across weather conditions.

In [12]:
# Create a pivot table for lap-by-lap position tracking

position_tracking_list = []

for race_id in df['RaceID'].unique():
    race_data = df[df['RaceID'] == race_id]
    event_name = race_data['EventName'].iloc[0]
    year = race_data['Year'].iloc[0]
    
    race_pivot = race_data.pivot_table(
        index=['Driver', 'Team', 'DriverNumber'],
        columns='LapNumber',
        values='Position',
        aggfunc='first'
    )
    
    # Race identifiers
    race_pivot['EventName'] = event_name
    race_pivot['Year'] = year
    race_pivot['RaceID'] = race_id
    
    position_tracking_list.append(race_pivot)

print(f"\nPosition tracking for first race:")
position_tracking_list[0].head(20)


Position tracking for first race:


,,LapNumber,1.0,2.0,3.0,4.0,5.0,8.0,9.0,10.0,11.0,12.0,...,51.0,52.0,53.0,54.0,55.0,56.0,57.0,EventName,Year,RaceID
Driver,Team,DriverNumber,,,,,,,,,,,,,,,,,,,,,
ALO,McLaren,14,10.0,10.0,10.0,10.0,NaN,10.0,10.0,NaN,10.0,10.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,Australian Grand Prix,2018,Australian Grand Prix 2018
BOT,Mercedes,77,15.0,15.0,15.0,14.0,NaN,14.0,NaN,13.0,13.0,13.0,...,8.0,8.0,8.0,8.0,8.0,8.0,8.0,Australian Grand Prix,2018,Australian Grand Prix 2018
ERI,Kick Sauber,9,16.0,16.0,16.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Australian Grand Prix,2018,Australian Grand Prix 2018
GAS,Racing Bulls,10,17.0,17.0,17.0,17.0,NaN,16.0,NaN,16.0,16.0,16.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Australian Grand Prix,2018,Australian Grand Prix 2018
GRO,Haas,8,6.0,6.0,6.0,6.0,NaN,6.0,6.0,NaN,5.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Australian Grand Prix,2018,Australian Grand Prix 2018
HAM,Mercedes,44,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,Australian Grand Prix,2018,Australian Grand Prix 2018
HAR,Racing Bulls,28,NaN,NaN,20.0,20.0,NaN,18.0,NaN,18.0,18.0,18.0,...,15.0,15.0,15.0,15.0,15.0,15.0,NaN,Australian Grand Prix,2018,Australian Grand Prix 2018
HUL,Alpine,27,7.0,7.0,7.0,7.0,NaN,8.0,8.0,NaN,7.0,7.0,...,7.0,7.0,7.0,7.0,7.0,7.0,7.0,Australian Grand Prix,2018,Australian Grand Prix 2018
LEC,Kick Sauber,16,18.0,18.0,18.0,18.0,NaN,17.0,NaN,17.0,17.0,17.0,...,13.0,13.0,13.0,13.0,13.0,13.0,13.0,Australian Grand Prix,2018,Australian Grand Prix 2018


## Position Change Analysis

Using the lap-by-lap position data, let us generate some more actionable information on driver overtakes

In [13]:
# dataframe tracking position changes within races
position_changes_list = []

for race_id in df['RaceID'].unique():
    race_data = df[df['RaceID'] == race_id]
    event_name = race_data['EventName'].iloc[0]
    year = race_data['Year'].iloc[0]
    location = race_data['Location'].iloc[0]
    
    for driver in race_data['Driver'].unique():
        driver_data = race_data[race_data['Driver'] == driver].sort_values('LapNumber')
        # Remove rows with NaN 
        driver_data = driver_data[driver_data['Position'].notna()]
        
        if len(driver_data) < 2:
            continue
        
        # Get driver info
        team = driver_data['Team'].iloc[0]
        driver_num = driver_data['DriverNumber'].iloc[0]
        
        # Track position changes lap by lap
        positions = driver_data['Position'].values
        lap_numbers = driver_data['LapNumber'].values
        
        # Calculate position changes
        for i in range(1, len(positions)):
            position_change = positions[i-1] - positions[i]  
            
            position_changes_list.append({
                'Driver': driver,
                'DriverNumber': driver_num,
                'Team': team,
                'EventName': event_name,
                'Year': year,
                'Location': location,
                'RaceID': race_id,
                'FromLap': int(lap_numbers[i-1]),
                'ToLap': int(lap_numbers[i]),
                'PositionBefore': positions[i-1],
                'PositionAfter': positions[i],
                'PositionChange': position_change
            })

position_changes_df = pd.DataFrame(position_changes_list)
print(f"position_changes_df: {position_changes_df.shape}")
print(f"\nFirst 10 position changes:")
position_changes_df.head(10)

position_changes_df: (152923, 12)

First 10 position changes:


,Driver,DriverNumber,Team,EventName,Year,Location,RaceID,FromLap,ToLap,PositionBefore,PositionAfter,PositionChange
0,GAS,10,Racing Bulls,Australian Grand Prix,2018,Melbourne,Australian Grand Prix 2018,1,2,17.0,17.0,0.0
1,GAS,10,Racing Bulls,Australian Grand Prix,2018,Melbourne,Australian Grand Prix 2018,2,3,17.0,17.0,0.0
2,GAS,10,Racing Bulls,Australian Grand Prix,2018,Melbourne,Australian Grand Prix 2018,3,4,17.0,17.0,0.0
3,GAS,10,Racing Bulls,Australian Grand Prix,2018,Melbourne,Australian Grand Prix 2018,4,8,17.0,16.0,1.0
4,GAS,10,Racing Bulls,Australian Grand Prix,2018,Melbourne,Australian Grand Prix 2018,8,10,16.0,16.0,0.0
5,GAS,10,Racing Bulls,Australian Grand Prix,2018,Melbourne,Australian Grand Prix 2018,10,11,16.0,16.0,0.0
6,GAS,10,Racing Bulls,Australian Grand Prix,2018,Melbourne,Australian Grand Prix 2018,11,12,16.0,16.0,0.0
7,GAS,10,Racing Bulls,Australian Grand Prix,2018,Melbourne,Australian Grand Prix 2018,12,13,16.0,16.0,0.0
8,PER,11,Aston Martin,Australian Grand Prix,2018,Melbourne,Australian Grand Prix 2018,1,2,12.0,12.0,0.0
9,PER,11,Aston Martin,Australian Grand Prix,2018,Melbourne,Australian Grand Prix 2018,2,3,12.0,12.0,0.0


In [14]:
print("\nMost dramatic position gains in single lap:")
print(position_changes_df.nlargest(10, 'PositionChange')[['Driver', 'EventName', 'Year', 'FromLap', 'ToLap', 'PositionChange']])

print("\nMost dramatic position losses in single lap:")
print(position_changes_df.nsmallest(10, 'PositionChange')[['Driver', 'EventName', 'Year', 'FromLap', 'ToLap', 'PositionChange']])



Most dramatic position gains in single lap:
       Driver                 EventName  Year  FromLap  ToLap  PositionChange
38578     MAG      Hungarian Grand Prix  2020        3      6            13.0
38944     GRO      Hungarian Grand Prix  2020        3      6            13.0
57726     STR        Styrian Grand Prix  2021        4      5            13.0
43080     GIO        Italian Grand Prix  2020       18     28            11.0
26808     NOR         German Grand Prix  2019        1      5            10.0
42957     RAI        Italian Grand Prix  2020       17     28            10.0
39755     GRO        British Grand Prix  2020       11     19             9.0
109884    ZHO  Saudi Arabian Grand Prix  2024        5     10             9.0
151424    OCO          Qatar Grand Prix  2025       32     33             9.0
26600     STR         German Grand Prix  2019        1      5             8.0

Most dramatic position losses in single lap:
      Driver             EventName  Year  FromLap  

#### Housekeeping Code

In [15]:
# Save the dataframes to CSV 
race_results_df.to_csv('../data/race_results_summary.csv', index=False)
driver_stats_df.to_csv('../data/driver_statistics.csv', index=False)
position_changes_df.to_csv('../data/position_changes.csv', index=False)


print(f"  - race_results_df: {race_results_df.shape}")
print(f"  - driver_stats_df: {driver_stats_df.shape}")
print(f"  - position_changes_df: {position_changes_df.shape}")

  - race_results_df: (3320, 17)
  - driver_stats_df: (43, 17)
  - position_changes_df: (152923, 12)



## Rainy Condition Analysis

The following sections replicate the race-level summary, driver statistics, and position change analysis from above, restricted to laps where **`Rainfall == True`**. Clean-lap filters are also applied (TrackStatus == 1, no pit laps, no terminal laps) to isolate meaningful racing laps in wet conditions.

### Filter to Rainy Clean Laps

Extract only laps where rainfall was detected (Rainfall=True) from races that had significant rain. This isolates rainy-condition performance data for drivers to enable weather-specific analysis.

In [16]:
# Filter to rainy, clean racing laps only
df_rain = df[(df['Rainfall'] == True)].copy()

df_rain['RaceID'] = df_rain['EventName'] + ' ' + df_rain['Year'].astype(str)

print(f"Total clean laps: {len(df)}")
print(f"Rainy clean laps: {len(df_rain)}")
print(f"Rainy races: {df_rain['RaceID'].nunique()}")
print(f"\nRainy races included:")
print(df_rain['RaceID'].unique())

Total clean laps: 156243
Rainy clean laps: 5525
Rainy races: 22

Rainy races included:
['Spanish Grand Prix 2018' 'Monaco Grand Prix 2018'
 'German Grand Prix 2018' 'Abu Dhabi Grand Prix 2018'
 'Monaco Grand Prix 2019' 'German Grand Prix 2019'
 'Emilia Romagna Grand Prix 2021' 'Belgian Grand Prix 2021'
 'Russian Grand Prix 2021' 'Monaco Grand Prix 2022'
 'Hungarian Grand Prix 2022' 'Japanese Grand Prix 2022'
 'Monaco Grand Prix 2023' 'Belgian Grand Prix 2023'
 'Dutch Grand Prix 2023' 'Canadian Grand Prix 2024'
 'Spanish Grand Prix 2024' 'British Grand Prix 2024'
 'São Paulo Grand Prix 2024' 'Australian Grand Prix 2025'
 'Miami Grand Prix 2025' 'British Grand Prix 2025']


### Race-Level Summary (Rainy Laps)

Create a race-level summary using only rainy laps, showing how drivers performed specifically during wet weather. This reveals which drivers are strong in rain and which struggle with adverse conditions.

In [17]:
rain_race_results_list = []

for race_id in df_rain['RaceID'].unique():
    race_data = df_rain[df_rain['RaceID'] == race_id]

    drivers = race_data[['Driver', 'DriverNumber', 'Team']].drop_duplicates()

    for _, driver_row in drivers.iterrows():
        driver = driver_row['Driver']
        driver_race = race_data[race_data['Driver'] == driver].copy()

        if len(driver_race) == 0:
            continue

        event_name = race_data['EventName'].iloc[0]
        year = race_data['Year'].iloc[0]
        location = race_data['Location'].iloc[0]

        valid_positions = driver_race['Position'].dropna()
        if len(valid_positions) == 0:
            continue

        first_lap = driver_race[driver_race['LapNumber'] == driver_race['LapNumber'].min()]
        grid_pos = first_lap['Position'].iloc[0] if len(first_lap) > 0 and pd.notna(first_lap['Position'].iloc[0]) else np.nan

        finishing_pos = valid_positions.iloc[-1]
        best_pos = valid_positions.min()
        worst_pos = valid_positions.max()
        avg_pos = valid_positions.mean()
        pos_std = valid_positions.std()

        positions_gained = (grid_pos - finishing_pos) if pd.notna(grid_pos) else np.nan
        laps_led = len(driver_race[driver_race['Position'] == 1.0])
        pit_stops = len(driver_race[driver_race['IsPitLap'] == True])  # always 0 after filter, kept for schema parity

        max_laps_any_driver = race_data['LapNumber'].max()
        driver_max_laps = driver_race['LapNumber'].max()
        dnf = driver_max_laps < (max_laps_any_driver - 2)

        rain_race_results_list.append({
            'Driver': driver,
            'DriverNumber': driver_row['DriverNumber'],
            'Team': driver_row['Team'],
            'EventName': event_name,
            'Year': year,
            'Location': location,
            'GridPosition': grid_pos,
            'FinishingPosition': finishing_pos,
            'BestPosition': best_pos,
            'WorstPosition': worst_pos,
            'AvgPosition': avg_pos,
            'PositionStdDev': pos_std,
            'PositionsGained': positions_gained,
            'LapsLed': laps_led,
            'RainyLapsCompleted': int(driver_max_laps),
            'DNF': dnf
        })

rain_race_results_df = pd.DataFrame(rain_race_results_list)

print(f"\nFirst race sample:")
rain_race_results_df.head(20)


First race sample:


,Driver,DriverNumber,Team,EventName,Year,Location,GridPosition,FinishingPosition,BestPosition,WorstPosition,AvgPosition,PositionStdDev,PositionsGained,LapsLed,RainyLapsCompleted,DNF
0,PER,11,Aston Martin,Spanish Grand Prix,2018,Barcelona,13.0,9.0,9.0,15.0,11.441860,1.776866,4.0,0,58,False
1,ALO,14,McLaren,Spanish Grand Prix,2018,Barcelona,10.0,8.0,8.0,15.0,9.690476,1.854304,2.0,0,58,False
2,LEC,16,Kick Sauber,Spanish Grand Prix,2018,Barcelona,9.0,10.0,8.0,15.0,9.619048,1.497191,-1.0,0,58,False
3,STR,18,Williams,Spanish Grand Prix,2018,Barcelona,12.0,11.0,10.0,17.0,12.139535,1.641448,1.0,0,58,False
4,VAN,2,McLaren,Spanish Grand Prix,2018,Barcelona,14.0,13.0,7.0,14.0,11.600000,2.711406,1.0,0,45,True
5,MAG,20,Haas,Spanish Grand Prix,2018,Barcelona,7.0,6.0,6.0,7.0,6.250000,0.438019,1.0,0,59,False
6,HAR,28,Racing Bulls,Spanish Grand Prix,2018,Barcelona,17.0,13.0,12.0,17.0,14.255814,1.733329,4.0,0,58,False
7,RIC,3,Red Bull Racing,Spanish Grand Prix,2018,Barcelona,6.0,5.0,3.0,6.0,4.711111,0.968181,1.0,0,59,False
8,OCO,31,Aston Martin,Spanish Grand Prix,2018,Barcelona,11.0,15.0,7.0,16.0,11.576923,2.995638,-4.0,0,38,True
9,VER,33,Red Bull Racing,Spanish Grand Prix,2018,Barcelona,5.0,3.0,1.0,5.0,3.088889,1.276042,2.0,8,59,False


### Driver-Level Statistics (Rainy Laps)

Compute driver-level statistics using only rainy laps, providing aggregate metrics (like average finish position in rain) that characterize each driver's wet-weather performance.

In [18]:
rain_driver_stats_list = []

for driver in rain_race_results_df['Driver'].unique():
    driver_races = rain_race_results_df[rain_race_results_df['Driver'] == driver]

    driver_num = driver_races['DriverNumber'].iloc[0]
    team = driver_races['Team'].iloc[0]

    races_entered = len(driver_races)
    races_completed = len(driver_races[driver_races['DNF'] == False])
    dnf_count = races_entered - races_completed

    avg_grid_pos = driver_races['GridPosition'].mean()
    avg_finish_pos = driver_races['FinishingPosition'].mean()
    avg_positions_gained = driver_races['PositionsGained'].mean()
    races_with_gains = len(driver_races[driver_races['PositionsGained'] > 0])
    races_with_losses = len(driver_races[driver_races['PositionsGained'] < 0])

    podiums = len(driver_races[driver_races['FinishingPosition'] <= 3])
    poles = len(driver_races[driver_races['GridPosition'] == 1.0])
    wins = len(driver_races[driver_races['FinishingPosition'] == 1.0])
    total_laps_led = driver_races['LapsLed'].sum()
    avg_position_std = driver_races['PositionStdDev'].mean()

    rain_driver_stats_list.append({
        'Driver': driver,
        'DriverNumber': driver_num,
        'Team': team,
        'RainyRacesEntered': races_entered,
        'RainyRacesCompleted': races_completed,
        'DNFCount': dnf_count,
        'DNFRate': dnf_count / races_entered if races_entered > 0 else 0,
        'AvgGridPosition': avg_grid_pos,
        'AvgFinishPosition': avg_finish_pos,
        'AvgPositionsGained': avg_positions_gained,
        'RacesWithGains': races_with_gains,
        'RacesWithLosses': races_with_losses,
        'Podiums': podiums,
        'Poles': poles,
        'Wins': wins,
        'TotalLapsLed': total_laps_led,
        'AvgPositionConsistency': avg_position_std
    })

rain_driver_stats_df = pd.DataFrame(rain_driver_stats_list).sort_values('RainyRacesEntered', ascending=False)
print(f"rain_driver_stats_df: {rain_driver_stats_df.shape}")
print(f"\nTop 10 drivers by rainy races entered:")
rain_driver_stats_df.head(10)

rain_driver_stats_df: (40, 17)

Top 10 drivers by rainy races entered:


,Driver,DriverNumber,Team,RainyRacesEntered,RainyRacesCompleted,DNFCount,DNFRate,AvgGridPosition,AvgFinishPosition,AvgPositionsGained,RacesWithGains,RacesWithLosses,Podiums,Poles,Wins,TotalLapsLed,AvgPositionConsistency
2,LEC,16,Kick Sauber,22,17,5,0.227273,7.545455,8.090909,-0.545455,7,6,5,1,0,8,1.093414
9,VER,33,Red Bull Racing,21,20,1,0.047619,4.047619,2.571429,1.476190,10,1,17,6,11,105,0.774278
3,STR,18,Williams,20,17,3,0.150000,13.100000,11.300000,1.800000,11,3,0,0,0,0,1.911043
11,HAM,44,Mercedes,20,18,2,0.100000,4.650000,4.300000,0.350000,4,2,9,5,5,125,0.976991
8,OCO,31,Aston Martin,20,15,5,0.250000,9.850000,9.250000,0.600000,7,4,2,0,0,3,1.058525
0,PER,11,Aston Martin,19,17,2,0.105263,9.947368,9.473684,0.473684,8,3,4,1,1,7,1.099211
1,ALO,14,McLaren,19,16,3,0.157895,9.052632,9.052632,0.000000,4,3,2,0,0,0,0.732421
15,BOT,77,Mercedes,19,15,4,0.210526,10.368421,9.157895,1.210526,10,4,3,0,0,0,1.217502
13,SAI,55,Alpine,19,16,3,0.157895,8.368421,7.421053,0.947368,10,1,2,0,0,2,0.775801
17,GAS,10,Racing Bulls,19,14,5,0.263158,10.210526,9.368421,0.842105,8,2,1,0,0,0,1.434341


In [19]:
print("Top 10 drivers by average positions gained in rain:")
print(rain_driver_stats_df.nlargest(10, 'AvgPositionsGained')[['Driver', 'Team', 'AvgPositionsGained', 'RainyRacesEntered']])

print("\nTop 10 drivers by race wins in rain:")
print(rain_driver_stats_df.nlargest(10, 'Wins')[['Driver', 'Team', 'Wins', 'Podiums', 'Poles']])

Top 10 drivers by average positions gained in rain:
   Driver             Team  AvgPositionsGained  RainyRacesEntered
20    KVY     Racing Bulls            8.000000                  2
24    KUB         Williams            4.000000                  2
12    VET          Ferrari            3.181818                 11
29    LAT         Williams            2.250000                  4
34    LAW     Racing Bulls            2.000000                  4
3     STR         Williams            1.800000                 20
9     VER  Red Bull Racing            1.476190                 21
22    NOR          McLaren            1.470588                 17
21    ALB     Racing Bulls            1.416667                 12
15    BOT         Mercedes            1.210526                 19

Top 10 drivers by race wins in rain:
   Driver             Team  Wins  Podiums  Poles
9     VER  Red Bull Racing    11       17      6
11    HAM         Mercedes     5        9      5
7     RIC  Red Bull Racing     2     

### Position Change Analysis (Rainy Laps)

Analyze how many positions drivers gain or lose during rainy races. This metric reveals overtaking aggression and adaptability in wet conditions—showing which drivers are better at managing rain-impacted races.

In [20]:
rain_position_changes_list = []

for race_id in df_rain['RaceID'].unique():
    race_data = df_rain[df_rain['RaceID'] == race_id]
    event_name = race_data['EventName'].iloc[0]
    year = race_data['Year'].iloc[0]
    location = race_data['Location'].iloc[0]

    for driver in race_data['Driver'].unique():
        driver_data = race_data[race_data['Driver'] == driver].sort_values('LapNumber')
        driver_data = driver_data[driver_data['Position'].notna()]

        if len(driver_data) < 2:
            continue

        team = driver_data['Team'].iloc[0]
        driver_num = driver_data['DriverNumber'].iloc[0]
        positions = driver_data['Position'].values
        lap_numbers = driver_data['LapNumber'].values

        for i in range(1, len(positions)):
            position_change = positions[i-1] - positions[i]
            rain_position_changes_list.append({
                'Driver': driver,
                'DriverNumber': driver_num,
                'Team': team,
                'EventName': event_name,
                'Year': year,
                'Location': location,
                'RaceID': race_id,
                'FromLap': int(lap_numbers[i-1]),
                'ToLap': int(lap_numbers[i]),
                'PositionBefore': positions[i-1],
                'PositionAfter': positions[i],
                'PositionChange': position_change
            })

rain_position_changes_df = pd.DataFrame(rain_position_changes_list)
print(f"rain_position_changes_df: {rain_position_changes_df.shape}")
print(f"\nFirst 10 rainy position changes:")
rain_position_changes_df.head(10)

rain_position_changes_df: (5121, 12)

First 10 rainy position changes:


,Driver,DriverNumber,Team,EventName,Year,Location,RaceID,FromLap,ToLap,PositionBefore,PositionAfter,PositionChange
0,PER,11,Aston Martin,Spanish Grand Prix,2018,Barcelona,Spanish Grand Prix 2018,10,11,13.0,13.0,0.0
1,PER,11,Aston Martin,Spanish Grand Prix,2018,Barcelona,Spanish Grand Prix 2018,11,12,13.0,13.0,0.0
2,PER,11,Aston Martin,Spanish Grand Prix,2018,Barcelona,Spanish Grand Prix 2018,12,13,13.0,13.0,0.0
3,PER,11,Aston Martin,Spanish Grand Prix,2018,Barcelona,Spanish Grand Prix 2018,13,14,13.0,13.0,0.0
4,PER,11,Aston Martin,Spanish Grand Prix,2018,Barcelona,Spanish Grand Prix 2018,14,15,13.0,13.0,0.0
5,PER,11,Aston Martin,Spanish Grand Prix,2018,Barcelona,Spanish Grand Prix 2018,15,16,13.0,13.0,0.0
6,PER,11,Aston Martin,Spanish Grand Prix,2018,Barcelona,Spanish Grand Prix 2018,16,17,13.0,13.0,0.0
7,PER,11,Aston Martin,Spanish Grand Prix,2018,Barcelona,Spanish Grand Prix 2018,17,18,13.0,13.0,0.0
8,PER,11,Aston Martin,Spanish Grand Prix,2018,Barcelona,Spanish Grand Prix 2018,18,19,13.0,13.0,0.0
9,PER,11,Aston Martin,Spanish Grand Prix,2018,Barcelona,Spanish Grand Prix 2018,19,20,13.0,12.0,1.0


In [21]:
print("Most dramatic position gains in a single rainy lap:")
print(rain_position_changes_df.nlargest(10, 'PositionChange')[['Driver', 'EventName', 'Year', 'FromLap', 'PositionChange']])

print("\nMost dramatic position losses in a single rainy lap:")
print(rain_position_changes_df.nsmallest(10, 'PositionChange')[['Driver', 'EventName', 'Year', 'FromLap', 'PositionChange']])

print(f"\nAverage position change per rainy lap: {rain_position_changes_df['PositionChange'].mean():.3f}")
print(f"Median position change per rainy lap: {rain_position_changes_df['PositionChange'].median():.3f}")

Most dramatic position gains in a single rainy lap:
     Driver                  EventName  Year  FromLap  PositionChange
2231    NOR          German Grand Prix  2019        1            10.0
2548    GAS  Emilia Romagna Grand Prix  2021       16            10.0
2027    STR          German Grand Prix  2019        1             8.0
3849    PIA           Dutch Grand Prix  2023        6             8.0
2049    STR          German Grand Prix  2019       46             7.0
2730    BOT         Russian Grand Prix  2021       49             7.0
2732    RAI         Russian Grand Prix  2021       49             7.0
3450    MSC        Japanese Grand Prix  2022        7             7.0
3762    STR           Dutch Grand Prix  2023       24             7.0
3817    LAW           Dutch Grand Prix  2023        3             7.0

Most dramatic position losses in a single rainy lap:
     Driver            EventName  Year  FromLap  PositionChange
3837    RUS     Dutch Grand Prix  2023        3           -1


## Rain vs. Dry Comparison

Comparing key driver metrics between rainy and dry conditions. The dry stats are derived from the overall `driver_stats_df` (all races) minus the rainy contributions approximated via `rain_driver_stats_df`. For a clean split, dry metrics are computed directly from laps where `Rainfall == False`.

In [22]:
# Build dry race-level results using same clean-lap logic
df_dry = df[
    (df['Rainfall'] == False)
    & (df['TrackStatus'] == 1)
    & (df['IsPitLap'] == False)
    & (df['IsTerminalLap'] == False)
].copy()
df_dry['RaceID'] = df_dry['EventName'] + ' ' + df_dry['Year'].astype(str)

dry_race_results_list = []

for race_id in df_dry['RaceID'].unique():
    race_data = df_dry[df_dry['RaceID'] == race_id]
    drivers = race_data[['Driver', 'DriverNumber', 'Team']].drop_duplicates()

    for _, driver_row in drivers.iterrows():
        driver = driver_row['Driver']
        driver_race = race_data[race_data['Driver'] == driver].copy()
        if len(driver_race) == 0:
            continue

        valid_positions = driver_race['Position'].dropna()
        if len(valid_positions) == 0:
            continue

        first_lap = driver_race[driver_race['LapNumber'] == driver_race['LapNumber'].min()]
        grid_pos = first_lap['Position'].iloc[0] if len(first_lap) > 0 and pd.notna(first_lap['Position'].iloc[0]) else np.nan
        finishing_pos = valid_positions.iloc[-1]
        positions_gained = (grid_pos - finishing_pos) if pd.notna(grid_pos) else np.nan

        max_laps_any = race_data['LapNumber'].max()
        dnf = driver_race['LapNumber'].max() < (max_laps_any - 2)

        dry_race_results_list.append({
            'Driver': driver,
            'DriverNumber': driver_row['DriverNumber'],
            'Team': driver_row['Team'],
            'FinishingPosition': finishing_pos,
            'GridPosition': grid_pos,
            'PositionsGained': positions_gained,
            'LapsLed': len(driver_race[driver_race['Position'] == 1.0]),
            'DNF': dnf
        })

dry_race_results_df = pd.DataFrame(dry_race_results_list)
print(f"dry_race_results_df: {dry_race_results_df.shape}")
dry_race_results_df.head()

dry_race_results_df: (3289, 8)


,Driver,DriverNumber,Team,FinishingPosition,GridPosition,PositionsGained,LapsLed,DNF
0,GAS,10,Racing Bulls,16.0,17.0,1.0,0,True
1,PER,11,Aston Martin,11.0,12.0,1.0,0,False
2,ALO,14,McLaren,5.0,10.0,5.0,0,False
3,LEC,16,Kick Sauber,13.0,18.0,5.0,0,False
4,STR,18,Williams,14.0,14.0,0.0,0,False


In [23]:
# Aggregate dry stats per driver
dry_driver_stats_df = (
    dry_race_results_df.groupby(['Driver']).agg(
        DryRacesEntered=('FinishingPosition', 'count'),
        DryAvgFinishPosition=('FinishingPosition', 'mean'),
        DryAvgPositionsGained=('PositionsGained', 'mean'),
        DryDNFCount=('DNF', 'sum'),
        DryWins=('FinishingPosition', lambda x: (x == 1).sum()),
        DryPodiums=('FinishingPosition', lambda x: (x <= 3).sum()),
        DryTotalLapsLed=('LapsLed', 'sum')
    )
    .reset_index()
)
dry_driver_stats_df['DryDNFRate'] = dry_driver_stats_df['DryDNFCount'] / dry_driver_stats_df['DryRacesEntered']
dry_driver_stats_df.head()
# Aggregate rain stats per driver (matching columns)
rain_agg = (
    rain_race_results_df.groupby('Driver')
    .agg(
        RainRacesEntered=('FinishingPosition', 'count'),
        RainAvgFinishPosition=('FinishingPosition', 'mean'),
        RainAvgPositionsGained=('PositionsGained', 'mean'),
        RainDNFCount=('DNF', 'sum'),
        RainWins=('FinishingPosition', lambda x: (x == 1).sum()),
        RainPodiums=('FinishingPosition', lambda x: (x <= 3).sum()),
        RainTotalLapsLed=('LapsLed', 'sum')
    )
    .reset_index()
)
rain_agg['RainDNFRate'] = rain_agg['RainDNFCount'] / rain_agg['RainRacesEntered']

# Merge rain and dry
comparison_df = rain_agg.merge(dry_driver_stats_df, on='Driver', how='inner')

# Key deltas
comparison_df['FinishPosDelta'] = comparison_df['RainAvgFinishPosition'] - comparison_df['DryAvgFinishPosition']
comparison_df['PositionsGainedDelta'] = comparison_df['RainAvgPositionsGained'] - comparison_df['DryAvgPositionsGained']
comparison_df['DNFRateDelta'] = comparison_df['RainDNFRate'] - comparison_df['DryDNFRate']

comparison_df[['Driver', 'RainAvgFinishPosition', 'DryAvgFinishPosition', 'FinishPosDelta',
                       'RainDNFRate', 'DryDNFRate', 'DNFRateDelta']].sort_values('FinishPosDelta').head(20)

,Driver,RainAvgFinishPosition,DryAvgFinishPosition,FinishPosDelta,RainDNFRate,DryDNFRate,DNFRateDelta
17,KVY,5.000000,10.947368,-5.947368,0.000000,0.131579,-0.131579
18,LAT,13.250000,15.706897,-2.456897,0.000000,0.189655,-0.189655
16,KUB,15.000000,17.434783,-2.434783,0.000000,0.304348,-0.304348
30,RUS,7.000000,8.946667,-1.946667,0.166667,0.106667,0.060000
11,GRO,11.200000,13.134615,-1.934615,0.000000,0.250000,-0.250000
38,VET,6.000000,7.887755,-1.887755,0.272727,0.091837,0.180891
28,RAI,8.500000,10.246753,-1.746753,0.125000,0.116883,0.008117
7,DEV,13.000000,14.636364,-1.636364,0.000000,0.090909,-0.090909
9,GAS,9.368421,10.728395,-1.359974,0.263158,0.098765,0.164392
6,COL,14.000000,15.250000,-1.250000,1.000000,0.041667,0.958333


In [24]:
# Drivers who finish better in rain than dry ( a negative FinishPosDelta indicates better position in rain)
print("Drivers who perform BETTER in rain (lower avg finish position):")
better_in_rain = comparison_df[comparison_df['FinishPosDelta'] < 0].sort_values('FinishPosDelta')
print(better_in_rain[['Driver', 'RainRacesEntered', 'RainAvgFinishPosition', 'DryAvgFinishPosition', 'FinishPosDelta']].to_string(index=False))

print("\nDrivers who perform WORSE in rain:")
worse_in_rain = comparison_df[comparison_df['FinishPosDelta'] > 0].sort_values('FinishPosDelta', ascending=False)
print(worse_in_rain[['Driver', 'RainRacesEntered', 'RainAvgFinishPosition', 'DryAvgFinishPosition', 'FinishPosDelta']].to_string(index=False))

Drivers who perform BETTER in rain (lower avg finish position):
Driver  RainRacesEntered  RainAvgFinishPosition  DryAvgFinishPosition  FinishPosDelta
   KVY                 2               5.000000             10.947368       -5.947368
   LAT                 4              13.250000             15.706897       -2.456897
   KUB                 2              15.000000             17.434783       -2.434783
   RUS                18               7.000000              8.946667       -1.946667
   GRO                 5              11.200000             13.134615       -1.934615
   VET                11               6.000000              7.887755       -1.887755
   RAI                 8               8.500000             10.246753       -1.746753
   DEV                 1              13.000000             14.636364       -1.636364
   GAS                19               9.368421             10.728395       -1.359974
   COL                 1              14.000000             15.250000       

In [25]:
# DNF rate comparison
print("DNF rate comparison — Rain vs Dry (sorted by rain DNF rate):")
print(
    comparison_df[['Driver', 'RainRacesEntered', 'RainDNFRate', 'DryDNFRate', 'DNFRateDelta']]
    .sort_values('RainDNFRate', ascending=False)
    .round(3)
    .to_string(index=False)
)

print("\nPositions gained comparison — Rain vs Dry:")
print(
    comparison_df[['Driver', 'RainAvgPositionsGained', 'DryAvgPositionsGained', 'PositionsGainedDelta']]
    .sort_values('PositionsGainedDelta', ascending=False)
    .round(3)
    .to_string(index=False)
)

DNF rate comparison — Rain vs Dry (sorted by rain DNF rate):
Driver  RainRacesEntered  RainDNFRate  DryDNFRate  DNFRateDelta
   COL                 1        1.000       0.042         0.958
   BOR                 2        0.500       0.095         0.405
   MSC                 5        0.400       0.195         0.205
   MAZ                 3        0.333       0.389        -0.056
   BEA                 3        0.333       0.111         0.222
   SIR                 3        0.333       0.158         0.175
   SAR                 6        0.333       0.194         0.139
   ZHO                10        0.300       0.164         0.136
   VET                11        0.273       0.092         0.181
   GAS                19        0.263       0.099         0.164
   VAN                 4        0.250       0.150         0.100
   OCO                20        0.250       0.106         0.144
   LEC                22        0.227       0.087         0.140
   BOT                19        0.211      


## Visualisations — Best Driver in Wet vs Normal Conditions

All charts use `comparison_df` and drivers are filtered to those with **≥ 5 rainy races** for statistical reliability.


In [26]:
import altair as alt

# Filter to drivers with enough rainy race data
MIN_RAIN_RACES = 5
comp = comparison_df[comparison_df['RainRacesEntered'] >= MIN_RAIN_RACES].copy()
comp = comp.sort_values('RainAvgFinishPosition')
print(f"Drivers with >= {MIN_RAIN_RACES} rainy races: {len(comp)}")
print(comp[['Driver', 'RainRacesEntered', 'RainAvgFinishPosition', 'DryAvgFinishPosition', 'FinishPosDelta']].to_string(index=False))

Drivers with >= 5 rainy races: 25
Driver  RainRacesEntered  RainAvgFinishPosition  DryAvgFinishPosition  FinishPosDelta
   VER                21               2.571429              2.780488       -0.209059
   HAM                20               4.300000              4.173653        0.126347
   PIA                 9               5.222222              5.867647       -0.645425
   VET                11               6.000000              7.887755       -1.887755
   NOR                17               6.294118              6.378378       -0.084261
   RUS                18               7.000000              8.946667       -1.946667
   SAI                19               7.421053              7.113208        0.307845
   LEC                22               8.090909              5.757764        2.333145
   RAI                 8               8.500000             10.246753       -1.746753
   ALO                19               9.052632              9.200000       -0.147368
   BOT              

#### Average Finish Position: Rain vs Dry

The bars are sorted by rainy finish position.

In [40]:
# Reshape to long format for grouped bar
finish_long = pd.melt(
    comp,
    id_vars=['Driver'],
    value_vars=['RainAvgFinishPosition', 'DryAvgFinishPosition'],
    var_name='Condition',
    value_name='AvgFinishPosition'
)
finish_long['Condition'] = finish_long['Condition'].map({
    'RainAvgFinishPosition': 'Rain',
    'DryAvgFinishPosition': 'Dry'
})

driver_order = comp['Driver'].tolist()

chart1 = alt.Chart(finish_long).mark_bar().encode(
    x=alt.X('Driver:N', sort=driver_order, title='Driver',
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('AvgFinishPosition:Q', title='Avg Finish Position',
            scale=alt.Scale(domain=[0, 20])),
    color=alt.Color('Condition:N',
                    scale=alt.Scale(domain=['Rain', 'Dry'],
                                    range=['#1f77b4', '#d62728']),
                    legend=alt.Legend(title='Condition')),
    xOffset='Condition:N',
    tooltip=['Driver', 'Condition', alt.Tooltip('AvgFinishPosition:Q', format='.2f')]
).properties(
    title='Average Finish Position — Rain vs Dry',
    width=700,
    height=350
).configure_title(fontSize=15, font='Calibri').configure_axis(labelFontSize=11)

chart1

alt.Chart(...)

#### Rain vs Dry Finish Position 

Drivers with points above the diagonal perform **worse** in raiin and drivers with points below perform **better** in rain.

In [44]:
# Diagonal reference line (y = x)
ref_line = alt.Chart(pd.DataFrame({'x': [1, 20], 'y': [1, 20]})).mark_line(
    color='gray', strokeDash=[4, 4], opacity=0.6
).encode(x='x:Q', y='y:Q')

scatter = alt.Chart(comp).mark_circle(size=90, opacity=0.85).encode(
    x=alt.X('DryAvgFinishPosition:Q', title='Dry Avg Finish Position',
            scale=alt.Scale(domain=[1, 18])),
    y=alt.Y('RainAvgFinishPosition:Q', title='Rain Avg Finish Position',
            scale=alt.Scale(domain=[1, 18])),
    color=alt.Color('Driver:N', legend=None), 
    tooltip=['Driver',
             alt.Tooltip('RainAvgFinishPosition:Q', format='.2f'),
             alt.Tooltip('DryAvgFinishPosition:Q', format='.2f'),
             alt.Tooltip('FinishPosDelta:Q', format='.2f', title='Rain Delta')]
)

labels = alt.Chart(comp).mark_text(dy=-10, fontSize=10).encode(
    x='DryAvgFinishPosition:Q',
    y='RainAvgFinishPosition:Q',
    text='Driver:N'
)

chart2 = (ref_line + scatter + labels).properties(
    title='Rain vs Dry Avg Finish Position',
    width=500,
    height=450
).configure_title(fontSize=14, font='Calibri')

chart2

alt.LayerChart(...)

#### Heatmap: Multi-Metric Driver Comparison

We will visualise drivers across six metrics which have been normalised normalised to the range of 0–1 (where **1 = best**). This gives us the chance to see how drivers are performing in specific metrics in comparision to other metrics

In [36]:
from sklearn.preprocessing import MinMaxScaler

hm_metrics = {
    'Rain Finish Pos':    ('RainAvgFinishPosition', True),   # True = lower is better (invert)
    'Dry Finish Pos':     ('DryAvgFinishPosition',  True),
    'Rain Pos Gained':    ('RainAvgPositionsGained', False),  # higher is better
    'Dry Pos Gained':     ('DryAvgPositionsGained',  False),
    'Rain DNF Rate':      ('RainDNFRate',            True),   # lower is better (invert)
    'Dry DNF Rate':       ('DryDNFRate',             True),
}

hm_rows = []
for label, (col, invert) in hm_metrics.items():
    vals = comp[col].values.astype(float)
    # Normalise 0-1
    mn, mx = vals.min(), vals.max()
    norm = (vals - mn) / (mx - mn) if mx > mn else vals * 0
    if invert:
        norm = 1 - norm
    for driver, score, raw in zip(comp['Driver'], norm, vals):
        hm_rows.append({'Driver': driver, 'Metric': label, 'Score': round(score, 3), 'RawValue': round(raw, 3)})

hm_df = pd.DataFrame(hm_rows)

# Sort drivers by rain finish position (best first)
driver_order_hm = comp.sort_values('RainAvgFinishPosition')['Driver'].tolist()
metric_order = list(hm_metrics.keys())

chart3 = alt.Chart(hm_df).mark_rect().encode(
    x=alt.X('Metric:N', sort=metric_order, title=None,
            axis=alt.Axis(labelAngle=-30, labelFontSize=11)),
    y=alt.Y('Driver:N', sort=driver_order_hm, title='Driver',
            axis=alt.Axis(labelFontSize=11)),
    color=alt.Color('Score:Q',
                    scale=alt.Scale(domain=[0, 1], range=['yellow', 'red']),
                    legend=alt.Legend(title='Score (1=best)')),
    tooltip=['Driver', 'Metric',
             alt.Tooltip('Score:Q', format='.3f'),
             alt.Tooltip('RawValue:Q', format='.3f', title='Raw Value')]
).properties(
    title='Driver Performance Heatmap - Rain vs Dry',
    width=840,
    height=420
).configure_title(fontSize=14, font='Calibri')

chart3

alt.Chart(...)

#### Finish Position Delta (Rain − Dry)

 We can use this to analyse which drivers are performing better in rain. Bars are sorted by delta and the more negative it is, the better their performance in the rain

In [43]:
delta_sorted = comp.sort_values('FinishPosDelta')

chart4 = alt.Chart(delta_sorted).mark_bar().encode(
    x=alt.X('Driver:N', sort=delta_sorted['Driver'].tolist(), title='Driver',
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('FinishPosDelta:Q',
            title='Finish Position Delta (Rain − Dry)',
            axis=alt.Axis(grid=True)),
    color=alt.condition(
        alt.datum.FinishPosDelta < 0,
        alt.value('#1f77b4'),   # blue = better in rain
        alt.value('#d62728')    # red = worse in rain
    ),
    tooltip=['Driver',
             alt.Tooltip('FinishPosDelta:Q', format='.2f', title='Delta (Rain−Dry)'),
             alt.Tooltip('RainAvgFinishPosition:Q', format='.2f'),
             alt.Tooltip('DryAvgFinishPosition:Q', format='.2f')]
).properties(
    title='Finish Position Delta — Rain vs Dry',
    width=840,
    height=320
).configure_title(fontSize=14, font='Calibri').configure_axis(labelFontSize=11)

chart4

alt.Chart(...)

#### Average Positions Gained: Rain vs Dry

 We can use this to analyse which drivers are gaining more positions in rain. The higher the value, the more they overtake better in the rain

In [31]:
gains_long = pd.melt(
    comp.sort_values('RainAvgPositionsGained', ascending=False),
    id_vars=['Driver'],
    value_vars=['RainAvgPositionsGained', 'DryAvgPositionsGained'],
    var_name='Condition',
    value_name='AvgPositionsGained'
)
gains_long['Condition'] = gains_long['Condition'].map({
    'RainAvgPositionsGained': 'Rain',
    'DryAvgPositionsGained': 'Dry'
})

gains_order = comp.sort_values('RainAvgPositionsGained', ascending=False)['Driver'].tolist()

chart5 = alt.Chart(gains_long).mark_bar().encode(
    x=alt.X('Driver:N', sort=gains_order, title='Driver',
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('AvgPositionsGained:Q', title='Avg Positions Gained per Race'),
    color=alt.Color('Condition:N',
                    scale=alt.Scale(domain=['Rain', 'Dry'],
                                    range=['#1f77b4', '#d62728']),
                    legend=alt.Legend(title='Condition')),
    xOffset='Condition:N',
    tooltip=['Driver', 'Condition', alt.Tooltip('AvgPositionsGained:Q', format='.2f')]
).properties(
    title='Avg Positions Gained per Race — Rain vs Dry',
    width=840,
    height=320
).configure_title(fontSize=14, font='Calibri').configure_axis(labelFontSize=11)

chart5

alt.Chart(...)